In [1]:
import pandas as pd
from dqeval.dataframe import DqEvalDataFrame
from dqeval.evals.unicode_validation_eval import UnicodeValidationEval

# Source table — mixed half-width katakana and replacement char (common ETL problems)
source_df = pd.DataFrame({
    "customer_id": [1,        2,          3,          4,       5],
    "name_jp":     ["田中太郎", "ｽｽﾞｷ花子",  "佐藤一郎",  "高橋\ufffd", "山田花子"],
    #                clean     half-width  clean       repl-char  clean
    "address_jp":  ["東京都",   "大阪府",    "京都府",    "北海道",    "福岡県"],
})

# Target table — clean, fully normalized values
target_df = pd.DataFrame({
    "customer_id": [1,        2,          3,          4,       5],
    "name_jp":     ["田中太郎", "スズキ花子",  "佐藤一郎",  "高橋",     "山田花子"],
    #                          full-width              clean (no repl)
    "address_jp":  ["東京都",   "大阪府",    "京都府",    "北海道",    "福岡県"],
})

result_json, failed_df = UnicodeValidationEval(
    DqEvalDataFrame(source_df),
    config={
        "key_column":          "customer_id",
        "columns":             ["name_jp", "address_jp"],
        "target_df":           target_df,
        "normalization_form":  "NFKC",   # ← critical for Japanese
        "batch_size":          1000,
    },
).run(evaluation="advanced")

import json
print(json.dumps(json.loads(result_json), indent=2, ensure_ascii=False))
print()
print(failed_df.to_string(index=False))

{
  "status": "Failed",
  "dqeval_type": "unicode_validation_eval",
  "key_column": "customer_id",
  "columns_checked": [
    "name_jp",
    "address_jp"
  ],
  "normalization_form": "NFKC",
  "dqeval_total_count": 5,
  "dqeval_matched_count": 5,
  "dqeval_failed_count": 1,
  "dqeval_passed_count": 4,
  "dqeval_mojibake_count": 0,
  "dqeval_replacement_char_count": 1,
  "run_timestamp": "2026-07-20T09:50:01.171869+00:00",
  "run_id": null
}

 customer_id name_jp address_jp
           4     高橋�        北海道
